[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/09-system-design/06_queues.ipynb)

*System Design · Notebook 6 of 10*

# Queues & Streams

Decoupling with messages: backlogs and consumer counts, Kafka partitions, keys and consumer groups, lag and offsets, at-least-once redelivery, retries with backoff and jitter, dead-letter queues, visibility timeouts, backpressure, the outbox pattern, reordering, windows and watermarks, log compaction, delay queues and queueing maths.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A queue is a restaurant's order rail: the waiter clips the ticket and moves on, and the kitchen works through tickets at its own pace.

**What interviewers really test here:** Partitions and ordering, consumer groups and lag, at-least-once delivery, retries and DLQs, backpressure, the outbox pattern and windows.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why put a queue in the middle? *(Basics)*
- [ ] 2. Kafka in one lesson *(Core)*
- [ ] 3. Delivery guarantees and failure handling *(Core)*
- [ ] 4. The outbox pattern and stream processing *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Queue vs log

|   | Queue (SQS, RabbitMQ) | Log (Kafka) |
| --- | --- | --- |
| After consuming | message deleted | kept for retention period |
| Replay | no | yes, reset the offset |
| Ordering | best effort (FIFO queues exist) | per partition |
| Many readers | competing consumers | independent consumer groups |

---
# Part 2: Lessons, step by step

## Lesson 1: Why put a queue in the middle?

*Level: Basics*

A queue **decouples** producers from consumers: the checkout service drops an "order placed" message and returns; email, inventory and analytics consume it at their own pace. Queues absorb spikes (the backlog grows, then drains), let you retry failed work, and let you scale consumers separately.



| Style | Examples | Model |
| --- | --- | --- |
| Message queue | SQS, RabbitMQ | each message to one consumer, deleted after ack |
| Log / stream | Kafka, Kinesis, Pub/Sub | append-only, partitioned, replayable; many consumer groups |
| Pub/sub | SNS, Redis pub/sub | each message to every subscriber |

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q9](#q9), [Q19](#q19). Try one now, or carry on and do them in Part 3.

## Lesson 2: Kafka in one lesson

*Level: Core*

A topic is split into **partitions**, each an ordered log. Producers pick a partition by key, so all events for `order-42` stay in order. Consumers in a **group** share partitions (one consumer per partition), and each records an **offset** — how far it has read. Lag = newest offset − committed offset is the key health metric. Producers **batch** messages for throughput (batch size + linger time). Compacted topics keep the latest value per key, acting like a table.

**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q10](#q10), [Q4](#q4), [Q20](#q20), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 3: Delivery guarantees and failure handling

*Level: Core*

- **At-most-once**: commit before processing — may lose messages.
- **At-least-once**: process, then commit — may process twice after a crash. The usual choice, with idempotent consumers.
- **Retries** use exponential backoff with jitter; after N failures move the message to a **dead-letter queue** for inspection.
- SQS hides a received message for a **visibility timeout**; if processing takes longer, it's delivered again.
- **Backpressure**: bound your queues and reject or slow producers rather than running out of memory.

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q5](#q5), [Q6](#q6), [Q12](#q12), [Q7](#q7), [Q13](#q13), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 4: The outbox pattern and stream processing

*Level: Advanced*

Writing to the database and publishing an event are two systems — one can succeed while the other fails. The **transactional outbox** writes the event to an outbox table in the same transaction as the change; a relay (or CDC such as Debezium) publishes it, retrying until it succeeds.



Stream processors (Flink, Kafka Streams, Spark) aggregate over **windows** by event time. Events arrive late and out of order, so a **watermark** says "we've probably seen everything up to time T"; later events are dropped or sent to a side output.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def backlog_demo():
    q = 0
    for second, arrivals in enumerate([80, 150, 220, 90, 40]):
        q = max(0, q + arrivals - 100)
        print(f"t={second}s arrivals={arrivals:3}  backlog={q}")


backlog_demo()

t=0s arrivals= 80  backlog=0
t=1s arrivals=150  backlog=50
t=2s arrivals=220  backlog=170
t=3s arrivals= 90  backlog=160
t=4s arrivals= 40  backlog=100


**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q15](#q15), [Q8](#q8), [Q16](#q16). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
from collections import Counter, defaultdict


<a id="q1"></a>
### Q1. Watch the Backlog Grow

🟢 Warm-up · *Arrivals vs consumption*

Messages arrive per second as listed; consumers process at most `rate` per second. Return the backlog at the end of each second.

**Examples**

- `arrivals = [80, 120, 200, 150, 60, 40, 30], rate = 100` → `[0, 20, 120, 170, 130, 70, 0]`

In [ ]:
def backlog(arrivals, rate):
    """
    Messages arrive per second as listed; consumers process at most rate per
    second. Return the backlog at the end of each second.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(backlog, [
    (([80, 120, 200, 150, 60, 40, 30], 100), [0, 20, 120, 170, 130, 70, 0]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with a running backlog.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.accumulate` with a max-0 step.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with a running backlog.

```python
def backlog(arrivals, rate):
    out, q = [], 0
    for a in arrivals:
        q = q + a - rate
        if q < 0:
            q = 0
        out.append(q)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.accumulate` with a max-0 step.

```python
def backlog(arrivals, rate):
    from itertools import accumulate
    return list(accumulate(arrivals, lambda q, a: max(0, q + a - rate), initial=0))[1:]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with a running backlog. That costs O(n) time and O(n) space."

"I can do better: `itertools.accumulate` with a max-0 step. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/watch-the-backlog-grow)

<a id="q2"></a>
### Q2. How Many Consumers and Partitions?

🟢 Warm-up · *Throughput sizing*

Messages arrive at `msgs_per_s`, each takes `ms_each` to process, and you keep `headroom` spare capacity. Return `(consumers needed, partitions needed)` — in Kafka each partition is read by at most one consumer in a group, so you need at least as many partitions.

**Examples**

- `msgs_per_s = 12000, ms_each = 5, headroom = 0.25` → `(80, 80)`
- `msgs_per_s = 300, ms_each = 200, headroom = 0.5` → `(120, 120)`

In [ ]:
def consumers_needed(msgs_per_s, ms_each, headroom):
    """
    Messages arrive at msgs_per_s, each takes ms_each to process, and you keep
    headroom spare capacity. Return (consumers needed, partitions needed) — in
    Kafka each partition is read by at most one consumer in a group, so you need
    at least as many partitions.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(consumers_needed, [
    ((12000, 5, 0.25), (80, 80)),
    ((300, 200, 0.5), (120, 120)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count up until capacity suffices.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`math.ceil`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count up until capacity suffices.

```python
def consumers_needed(msgs_per_s, ms_each, headroom):
    per_consumer = 1000 / ms_each
    c = 0
    while c * per_consumer * (1 - headroom) < msgs_per_s - 1e-9:
        c += 1
    return (c, c)
```

**Cost:** O(c) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `math.ceil`.

```python
def consumers_needed(msgs_per_s, ms_each, headroom):
    c = math.ceil(round(msgs_per_s * ms_each / 1000 / (1 - headroom), 9))
    return (c, c)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count up until capacity suffices. That costs O(c) time and O(1) space."

"I can do better: `math.ceil`. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/how-many-consumers-and-partitions)

<a id="q3"></a>
### Q3. Partition by Key

🟢 Warm-up · *Order per key*

Send each `(key, event)` to partition `crc32(key) % p`. Return each partition's events in order. All events for one key stay in one partition, so their order is kept.

**Examples**

- `events = [('order-1', 'created'), ('order-2', 'created'), ('order-1', 'paid'), ('order-3', 'created'), ('order-2', 'cancelled'), ('order-1', 'shipped')], p = 3` → `[['order-2:created', 'order-2:cancelled'], ['order-1:created', 'order-1:paid', 'order-1:shipped'], ['order-3:created']]`

In [ ]:
import zlib

def partition_events(events, p):
    """
    Send each (key, event) to partition crc32(key) % p. Return each partition's
    events in order. All events for one key stay in one partition, so their
    order is kept.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partition_events, [
    (([('order-1', 'created'), ('order-2', 'created'), ('order-1', 'paid'), ('order-3', 'created'), ('order-2', 'cancelled'), ('order-1', 'shipped')], 3), [['order-2:created', 'order-2:cancelled'], ['order-1:created', 'order-1:paid', 'order-1:shipped'], ['order-3:created']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with a comprehension per partition.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list of lists.

```python
def partition_events(events, p):
    parts = [[] for _ in range(p)]
    for key, ev in events:
        parts[zlib.crc32(key.encode()) % p].append(key + ":" + ev)
    return parts
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with a comprehension per partition.

```python
def partition_events(events, p):
    return [[f"{k}:{e}" for k, e in events if zlib.crc32(k.encode()) % p == i] for i in range(p)]
```

**Cost:** O(n · p) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of lists. That costs O(n) time and O(n) space."

"I can do better: Same with a comprehension per partition. That's O(n · p) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/partition-by-key)

<a id="q4"></a>
### Q4. Consumer Lag

🟢 Warm-up · *How far behind are we?*

Lag is the newest offset minus the committed offset per partition. Return the lag per partition, the total, and the partitions lagging more than `alert`.

**Examples**

- `end_offsets = [1500, 980, 2200, 760], committed = [1490, 400, 2200, 100], alert = 500` → `([10, 580, 0, 660], 1250, [1, 3])`

In [ ]:
def consumer_lag(end_offsets, committed, alert):
    """
    Lag is the newest offset minus the committed offset per partition. Return
    the lag per partition, the total, and the partitions lagging more than
    alert.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(consumer_lag, [
    (([1500, 980, 2200, 760], [1490, 400, 2200, 100], 500), ([10, 580, 0, 660], 1250, [1, 3])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with three outputs.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`zip` and comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with three outputs.

```python
def consumer_lag(end_offsets, committed, alert):
    lags, hot = [], []
    for i in range(len(end_offsets)):
        lag = end_offsets[i] - committed[i]
        lags.append(lag)
        if lag > alert:
            hot.append(i)
    return (lags, sum(lags), hot)
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `zip` and comprehensions.

```python
def consumer_lag(end_offsets, committed, alert):
    lags = [e - c for e, c in zip(end_offsets, committed)]
    return (lags, sum(lags), [i for i, l in enumerate(lags) if l > alert])
```

**Cost:** O(p) time, O(p) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with three outputs. That costs O(p) time and O(p) space."

"I can do better: `zip` and comprehensions. That's O(p) time and O(p) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/consumer-lag)

<a id="q5"></a>
### Q5. Exponential Backoff

🟢 Warm-up · *Don't hammer a struggling service*

Return the wait (ms) before each retry: `min(cap, base × factor^attempt)` for attempts 0..n-1.

**Examples**

- `base = 100, factor = 2, cap = 5000, n = 8` → `[100, 200, 400, 800, 1600, 3200, 5000, 5000]`

In [ ]:
def backoff(base, factor, cap, n):
    """
    Return the wait (ms) before each retry: min(cap, base × factor^attempt) for
    attempts 0..n-1.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(backoff, [
    ((100, 2, 5000, 8), [100, 200, 400, 800, 1600, 3200, 5000, 5000]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop multiplying the delay.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop multiplying the delay.

```python
def backoff(base, factor, cap, n):
    out, d = [], base
    for _ in range(n):
        out.append(min(cap, d))
        d *= factor
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def backoff(base, factor, cap, n):
    return [min(cap, base * factor ** i) for i in range(n)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop multiplying the delay. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/exponential-backoff)

<a id="q6"></a>
### Q6. Add Full Jitter

🟢 Warm-up · *Spread the retries*

Without jitter, every client that failed together retries together. Full jitter waits `random × min(cap, base × 2^attempt)`. Using the given random numbers (one per client, attempt 3), return each client's wait in ms (rounded) and how many distinct wait times there are without and with jitter.

**Examples**

- `base = 100, cap = 5000, attempt = 3, randoms = [0.12, 0.87, 0.45, 0.99, 0.03, 0.61]` → `([96, 696, 360, 792, 24, 488], 1, 6)`

In [ ]:
def full_jitter(base, cap, attempt, randoms):
    """
    Without jitter, every client that failed together retries together. Full
    jitter waits random × min(cap, base × 2^attempt). Using the given random
    numbers (one per client, attempt 3), return each client's wait in ms
    (rounded) and how many distinct wait times there are without and with
    jitter.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(full_jitter, [
    ((100, 5000, 3, [0.12, 0.87, 0.45, 0.99, 0.03, 0.61]), ([96, 696, 360, 792, 24, 488], 1, 6)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension and sets.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop.

```python
def full_jitter(base, cap, attempt, randoms):
    ceiling = min(cap, base * 2 ** attempt)
    waits = []
    for r in randoms:
        waits.append(round(r * ceiling))
    return (waits, 1, len(set(waits)))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension and sets.

```python
def full_jitter(base, cap, attempt, randoms):
    ceiling = min(cap, base * 2 ** attempt)
    waits = [round(r * ceiling) for r in randoms]
    return (waits, 1, len(set(waits)))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop. That costs O(n) time and O(n) space."

"I can do better: A comprehension and sets. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/add-full-jitter)

<a id="q7"></a>
### Q7. Visibility Timeouts

🟢 Warm-up · *SQS-style redelivery*

A received message is hidden for `timeout` seconds; if not deleted in time it becomes visible and another consumer gets it too. For each processing time, return how many times the message gets delivered.

**Examples**

- `durations = [5, 30, 31, 75, 120], timeout = 30` → `[1, 1, 2, 3, 4]`

In [ ]:
def deliveries(durations, timeout):
    """
    A received message is hidden for timeout seconds; if not deleted in time it
    becomes visible and another consumer gets it too. For each processing time,
    return how many times the message gets delivered.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(deliveries, [
    (([5, 30, 31, 75, 120], 30), [1, 1, 2, 3, 4]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count deliveries by stepping through time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Ceiling division.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count deliveries by stepping through time.

```python
def deliveries(durations, timeout):
    out = []
    for d in durations:
        count, visible_again = 1, timeout
        while visible_again < d:
            count += 1
            visible_again += timeout
        out.append(count)
    return out
```

**Cost:** O(n · d) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Ceiling division.

```python
def deliveries(durations, timeout):
    return [max(1, math.ceil(d / timeout)) for d in durations]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count deliveries by stepping through time. That costs O(n · d) time and O(n) space."

"I can do better: Ceiling division. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/visibility-timeouts)

<a id="q8"></a>
### Q8. Tumbling Window Counts

🟢 Warm-up · *Stream aggregation*

Group events `(timestamp_s, amount)` into fixed windows of `size` seconds. Return `(window start, total)` sorted by window.

**Examples**

- `events = [(3, 10), (61, 5), (59, 7), (125, 1), (64, 2), (0, 4)], size = 60` → `[(0, 21), (60, 7), (120, 1)]`

In [ ]:
def tumbling(events, size):
    """
    Group events (timestamp_s, amount) into fixed windows of size seconds.
    Return (window start, total) sorted by window.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(tumbling, [
    (([(3, 10), (61, 5), (59, 7), (125, 1), (64, 2), (0, 4)], 60), [(0, 21), (60, 7), (120, 1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A dict of totals.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict` and sorted items.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A dict of totals.

```python
def tumbling(events, size):
    totals = {}
    for ts, amount in events:
        start = ts - ts % size
        totals[start] = totals.get(start, 0) + amount
    return sorted(totals.items())
```

**Cost:** O(n log n) time, O(w) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `defaultdict` and sorted items.

```python
def tumbling(events, size):
    totals = defaultdict(int)
    for ts, amount in events:
        totals[ts // size * size] += amount
    return sorted(totals.items())
```

**Cost:** O(n log n) time, O(w) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A dict of totals. That costs O(n log n) time and O(w) space."

"I can do better: `defaultdict` and sorted items. That's O(n log n) time and O(w) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/tumbling-window-counts)

<a id="q9"></a>
### Q9. Pub/Sub Fan-Out

🟢 Warm-up · *One message, many subscribers*

Subscribers listen to topics. For the published messages (topic names), return how many messages each subscriber receives and the total deliveries.

**Examples**

- `subs = {'email': ['orders', 'refunds'], 'analytics': ['orders', 'clicks', 'refunds'], 'fraud': ['orders']}, messages = ['orders', 'clicks', 'orders', 'refunds', 'clicks']` → `({'email': 3, 'analytics': 5, 'fraud': 2}, 10)`

In [ ]:
def fan_out(subs, messages):
    """
    Subscribers listen to topics. For the published messages (topic names),
    return how many messages each subscriber receives and the total deliveries.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fan_out, [
    (({'email': ['orders', 'refunds'], 'analytics': ['orders', 'clicks', 'refunds'], 'fraud': ['orders']}, ['orders', 'clicks', 'orders', 'refunds', 'clicks']), ({'email': 3, 'analytics': 5, 'fraud': 2}, 10)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each message, loop over subscribers.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count messages per topic first.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** For each message, loop over subscribers.

```python
def fan_out(subs, messages):
    counts = {s: 0 for s in subs}
    for m in messages:
        for s, topics in subs.items():
            if m in topics:
                counts[s] += 1
    return (counts, sum(counts.values()))
```

**Cost:** O(m · s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Count messages per topic first.

```python
def fan_out(subs, messages):
    per_topic = Counter(messages)
    counts = {s: sum(per_topic[t] for t in topics) for s, topics in subs.items()}
    return (counts, sum(counts.values()))
```

**Cost:** O(m + s · t) time, O(t) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each message, loop over subscribers. That costs O(m · s) time and O(s) space."

"I can do better: Count messages per topic first. That's O(m + s · t) time and O(t) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/pub-sub-fan-out)

<a id="q10"></a>
### Q10. Assign Partitions to a Consumer Group

🟡 Interview standard · *Range assignment*

Split partitions 0..n-1 among the sorted consumers in contiguous ranges; the first `n % c` consumers get one extra. Return the mapping. Extra consumers beyond n get nothing — they sit idle.

**Examples**

- `n = 7, consumers = ['c2', 'c1', 'c3']` → `{'c1': [0, 1, 2], 'c2': [3, 4], 'c3': [5, 6]}`
- `n = 2, consumers = ['a', 'b', 'c']` → `{'a': [0], 'b': [1], 'c': []}`

In [ ]:
def range_assign(n, consumers):
    """
    Split partitions 0..n-1 among the sorted consumers in contiguous ranges; the
    first n % c consumers get one extra. Return the mapping. Extra consumers
    beyond n get nothing — they sit idle.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(range_assign, [
    ((7, ['c2', 'c1', 'c3']), {'c1': [0, 1, 2], 'c2': [3, 4], 'c3': [5, 6]}),
    ((2, ['a', 'b', 'c']), {'a': [0], 'b': [1], 'c': []}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Hand out partitions one consumer at a time.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`divmod` for sizes, then slices.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Hand out partitions one consumer at a time.

```python
def range_assign(n, consumers):
    consumers = sorted(consumers)
    out, nxt = {}, 0
    for i, c in enumerate(consumers):
        size = n // len(consumers)
        if i < n % len(consumers):
            size += 1
        out[c] = list(range(nxt, nxt + size))
        nxt += size
    return out
```

**Cost:** O(n + c) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `divmod` for sizes, then slices.

```python
def range_assign(n, consumers):
    q, r = divmod(n, len(consumers))
    out, start = {}, 0
    for i, c in enumerate(sorted(consumers)):
        end = start + q + (i < r)
        out[c], start = list(range(start, end)), end
    return out
```

**Cost:** O(n + c) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Hand out partitions one consumer at a time. That costs O(n + c) time and O(n) space."

"I can do better: `divmod` for sizes, then slices. That's O(n + c) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/assign-partitions-to-a-consumer-group)

<a id="q11"></a>
### Q11. Duplicates After a Crash

🟡 Interview standard · *At-least-once delivery*

A consumer processes messages 0..n-1 and commits its offset after every `commit_every` messages. It crashes right after processing message `crash_at`, restarts from the last commit and finishes. Return `(messages processed twice, total processing runs)`.

**Examples**

- `n = 100, commit_every = 10, crash_at = 57` → `(8, 108)`
- `n = 100, commit_every = 1, crash_at = 57` → `(0, 100)`
- `n = 100, commit_every = 50, crash_at = 49` → `(0, 100)`

In [ ]:
def crash_duplicates(n, commit_every, crash_at):
    """
    A consumer processes messages 0..n-1 and commits its offset after every
    commit_every messages. It crashes right after processing message crash_at,
    restarts from the last commit and finishes. Return (messages processed
    twice, total processing runs).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(crash_duplicates, [
    ((100, 10, 57), (8, 108)),
    ((100, 1, 57), (0, 100)),
    ((100, 50, 49), (0, 100)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate processing and commits.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Arithmetic on the last commit point.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate processing and commits.

```python
def crash_duplicates(n, commit_every, crash_at):
    committed, processed = 0, 0
    for i in range(crash_at + 1):
        processed += 1
        if (i + 1) % commit_every == 0:
            committed = i + 1
    for i in range(committed, n):
        processed += 1
    return (processed - n, processed)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Arithmetic on the last commit point.

```python
def crash_duplicates(n, commit_every, crash_at):
    committed = (crash_at + 1) // commit_every * commit_every
    dup = crash_at + 1 - committed
    return (dup, n + dup)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate processing and commits. That costs O(n) time and O(1) space."

"I can do better: Arithmetic on the last commit point. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/duplicates-after-a-crash)

<a id="q12"></a>
### Q12. Send Poison Messages to a Dead-Letter Queue

🟡 Interview standard · *Don't retry forever*

Each message fails a given number of times before it would succeed (a large number means it never will). Try each at most `max_attempts` times. Return `(succeeded, dead-lettered, total attempts)`.

**Examples**

- `failures = {'m1': 0, 'm2': 2, 'm3': 99, 'm4': 1, 'm5': 5}, max_attempts = 3` → `(['m1', 'm2', 'm4'], ['m3', 'm5'], 12)`

In [ ]:
def dead_letter(failures, max_attempts):
    """
    Each message fails a given number of times before it would succeed (a large
    number means it never will). Try each at most max_attempts times. Return
    (succeeded, dead-lettered, total attempts).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dead_letter, [
    (({'m1': 0, 'm2': 2, 'm3': 99, 'm4': 1, 'm5': 5}, 3), (['m1', 'm2', 'm4'], ['m3', 'm5'], 12)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate attempts per message.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compare failures with max attempts directly.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate attempts per message.

```python
def dead_letter(failures, max_attempts):
    ok, dlq, attempts = [], [], 0
    for m, f in failures.items():
        tries = 0
        done = False
        while tries < max_attempts and not done:
            tries += 1
            if tries > f:
                done = True
        attempts += tries
        if done:
            ok.append(m)
        else:
            dlq.append(m)
    return (ok, dlq, attempts)
```

**Cost:** O(n · a) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Compare failures with max attempts directly.

```python
def dead_letter(failures, max_attempts):
    ok = [m for m, f in failures.items() if f < max_attempts]
    dlq = [m for m, f in failures.items() if f >= max_attempts]
    return (ok, dlq, sum(min(f + 1, max_attempts) for f in failures.values()))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate attempts per message. That costs O(n · a) time and O(n) space."

"I can do better: Compare failures with max attempts directly. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/send-poison-messages-to-a-dead-letter-queue)

<a id="q13"></a>
### Q13. Backpressure With a Bounded Queue

🟡 Interview standard · *Reject early, stay healthy*

Each tick, the listed number of requests arrive at a queue of `capacity` (extras are rejected with 429/503), then `service` are processed. Return `(rejected total, deepest queue)`.

**Examples**

- `arrivals = [5, 12, 20, 3, 0, 15], capacity = 10, service = 6` → `(21, 10)`

In [ ]:
def bounded_queue(arrivals, capacity, service):
    """
    Each tick, the listed number of requests arrive at a queue of capacity
    (extras are rejected with 429/503), then service are processed. Return
    (rejected total, deepest queue).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(bounded_queue, [
    (([5, 12, 20, 3, 0, 15], 10, 6), (21, 10)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with a counter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with `min`/`max`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with a counter.

```python
def bounded_queue(arrivals, capacity, service):
    q, rejected, deepest = 0, 0, 0
    for a in arrivals:
        space = capacity - q
        if a > space:
            rejected += a - space
            q = capacity
        else:
            q += a
        if q > deepest:
            deepest = q
        q -= service
        if q < 0:
            q = 0
    return (rejected, deepest)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with `min`/`max`.

```python
def bounded_queue(arrivals, capacity, service):
    q = rejected = deepest = 0
    for a in arrivals:
        take = min(a, capacity - q)
        rejected, q = rejected + a - take, q + take
        deepest, q = max(deepest, q), max(0, q - service)
    return (rejected, deepest)
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with a counter. That costs O(n) time and O(1) space."

"I can do better: Same with `min`/`max`. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/backpressure-with-a-bounded-queue)

<a id="q14"></a>
### Q14. Dual Writes vs the Outbox

🟡 Interview standard · *Never lose an event*

Each order is `(id, db_ok, publish_ok)`. Writing the DB then publishing loses the event when the DB write succeeds but publishing fails; publishing first creates 'phantom' events when the DB write then fails. With the outbox pattern the event is saved in the same DB transaction and a relay retries publishing, so neither happens. Return `(lost, phantom, outbox problems)`.

**Examples**

- `orders = [('o1', True, True), ('o2', True, False), ('o3', False, True), ('o4', True, False), ('o5', False, False)]` → `(['o2', 'o4'], ['o3'], 0)`

In [ ]:
def outbox(orders):
    """
    Each order is (id, db_ok, publish_ok). Writing the DB then publishing loses
    the event when the DB write succeeds but publishing fails; publishing first
    creates 'phantom' events when the DB write then fails. With the outbox
    pattern the event is saved in the same DB transaction and a relay retries
    publishing, so neither happens. Return (lost, phantom, outbox problems).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(outbox, [
    (([('o1', True, True), ('o2', True, False), ('o3', False, True), ('o4', True, False), ('o5', False, False)],), (['o2', 'o4'], ['o3'], 0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop with if-statements.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehensions.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop with if-statements.

```python
def outbox(orders):
    lost, phantom = [], []
    for oid, db_ok, pub_ok in orders:
        if db_ok and not pub_ok:
            lost.append(oid)
        if pub_ok and not db_ok:
            phantom.append(oid)
    return (lost, phantom, 0)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehensions.

```python
def outbox(orders):
    return ([o for o, d, p in orders if d and not p], [o for o, d, p in orders if p and not d], 0)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop with if-statements. That costs O(n) time and O(n) space."

"I can do better: Comprehensions. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/dual-writes-vs-the-outbox)

<a id="q15"></a>
### Q15. Restore Order With a Reorder Buffer

🟡 Interview standard · *Release in sequence*

Messages with sequence numbers 1, 2, 3, ... arrive out of order. Hold them until the next expected one arrives, then release every consecutive message. Return what is released after each arrival.

**Examples**

- `arrivals = [3, 1, 2, 6, 4, 5, 7]` → `[[], [1], [2, 3], [], [4], [5, 6], [7]]`

In [ ]:
def reorder(arrivals):
    """
    Messages with sequence numbers 1, 2, 3, ... arrive out of order. Hold them
    until the next expected one arrives, then release every consecutive message.
    Return what is released after each arrival.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reorder, [
    (([3, 1, 2, 6, 4, 5, 7],), [[], [1], [2, 3], [], [4], [5, 6], [7]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Keep a list; scan it for the next expected number.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of held numbers and a pointer.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Keep a list; scan it for the next expected number.

```python
def reorder(arrivals):
    held, nxt, out = [], 1, []
    for s in arrivals:
        held.append(s)
        released = []
        while nxt in held:
            held.remove(nxt)
            released.append(nxt)
            nxt += 1
        out.append(released)
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of held numbers and a pointer.

```python
def reorder(arrivals):
    held, nxt, out = set(), 1, []
    for s in arrivals:
        held.add(s)
        start = nxt
        while nxt in held:
            held.discard(nxt)
            nxt += 1
        out.append(list(range(start, nxt)))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Keep a list; scan it for the next expected number. That costs O(n²) time and O(n) space."

"I can do better: A set of held numbers and a pointer. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/restore-order-with-a-reorder-buffer)

<a id="q16"></a>
### Q16. Drop Late Events With a Watermark

🟡 Interview standard · *Event time vs arrival order*

Events arrive in the listed order carrying their event time. The watermark is the highest event time seen so far minus `lateness`; an event older than the current watermark is too late and is dropped. Return the dropped event times.

**Examples**

- `times = [10, 12, 11, 20, 14, 16, 25, 21, 9], lateness = 5` → `[14, 9]`

In [ ]:
def late_events(times, lateness):
    """
    Events arrive in the listed order carrying their event time. The watermark
    is the highest event time seen so far minus lateness; an event older than
    the current watermark is too late and is dropped. Return the dropped event
    times.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(late_events, [
    (([10, 12, 11, 20, 14, 16, 25, 21, 9], 5), [14, 9]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track the max seen in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, as a single pass.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track the max seen in a loop.

```python
def late_events(times, lateness):
    highest, dropped = None, []
    for t in times:
        if highest is not None and t < highest - lateness:
            dropped.append(t)
        else:
            if highest is None or t > highest:
                highest = t
    return dropped
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, as a single pass.

```python
def late_events(times, lateness):
    highest, dropped = -math.inf, []
    for t in times:
        if t < highest - lateness:
            dropped.append(t)
        else:
            highest = max(highest, t)
    return dropped
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track the max seen in a loop. That costs O(n) time and O(n) space."

"I can do better: Same, as a single pass. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/drop-late-events-with-a-watermark)

<a id="q17"></a>
### Q17. A Delay Queue

🟡 Interview standard · *Run jobs at a scheduled time*

Jobs `(run_at, name)` are added in any order. A worker wakes at each time in `ticks` and runs every due job (earliest first, ties by name). Return `(tick, [jobs run])` for ticks that ran something.

**Examples**

- `jobs = [(30, 'reminder'), (5, 'retry-7'), (12, 'cleanup'), (5, 'retry-3'), (31, 'report')], ticks = [0, 10, 20, 30, 40]` → `[(10, ['retry-3', 'retry-7']), (20, ['cleanup']), (30, ['reminder']), (40, ['report'])]`

In [ ]:
import heapq

def delay_queue(jobs, ticks):
    """
    Jobs (run_at, name) are added in any order. A worker wakes at each time in
    ticks and runs every due job (earliest first, ties by name). Return (tick,
    [jobs run]) for ticks that ran something.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(delay_queue, [
    (([(30, 'reminder'), (5, 'retry-7'), (12, 'cleanup'), (5, 'retry-3'), (31, 'report')], [0, 10, 20, 30, 40]), [(10, ['retry-3', 'retry-7']), (20, ['cleanup']), (30, ['reminder']), (40, ['report'])]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort the remaining jobs at every tick.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A min-heap.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort the remaining jobs at every tick.

```python
def delay_queue(jobs, ticks):
    pending, out = list(jobs), []
    for t in ticks:
        pending.sort()
        due = [j for j in pending if j[0] <= t]
        pending = [j for j in pending if j[0] > t]
        if due:
            out.append((t, [name for _, name in due]))
    return out
```

**Cost:** O(t · n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A min-heap.

```python
def delay_queue(jobs, ticks):
    heap, out = list(jobs), []
    heapq.heapify(heap)
    for t in ticks:
        ran = []
        while heap and heap[0][0] <= t:
            ran.append(heapq.heappop(heap)[1])
        if ran:
            out.append((t, ran))
    return out
```

**Cost:** O(n log n + t) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort the remaining jobs at every tick. That costs O(t · n log n) time and O(n) space."

"I can do better: A min-heap. That's O(n log n + t) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/a-delay-queue)

<a id="q18"></a>
### Q18. Log Compaction

🟡 Interview standard · *Keep the latest value per key*

A compacted topic keeps only the newest record per key (a None value is a delete and is removed too). Records are `(key, value)` at offsets 0, 1, 2…; return the surviving `(offset, key, value)` in offset order.

**Examples**

- `records = [('u1', 'Pune'), ('u2', 'Goa'), ('u1', 'Delhi'), ('u3', 'Agra'), ('u2', None), ('u1', 'Mumbai')]` → `[(3, 'u3', 'Agra'), (5, 'u1', 'Mumbai')]`

In [ ]:
def compact_log(records):
    """
    A compacted topic keeps only the newest record per key (a None value is a
    delete and is removed too). Records are (key, value) at offsets 0, 1, 2…;
    return the surviving (offset, key, value) in offset order.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(compact_log, [
    (([('u1', 'Pune'), ('u2', 'Goa'), ('u1', 'Delhi'), ('u3', 'Agra'), ('u2', None), ('u1', 'Mumbai')],), [(3, 'u3', 'Agra'), (5, 'u1', 'Mumbai')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each record, check whether a later one has the same key.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Remember the last offset per key in one pass.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** For each record, check whether a later one has the same key.

```python
def compact_log(records):
    out = []
    for i, (k, v) in enumerate(records):
        later = False
        for j in range(i + 1, len(records)):
            if records[j][0] == k:
                later = True
        if not later and v is not None:
            out.append((i, k, v))
    return out
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Remember the last offset per key in one pass.

```python
def compact_log(records):
    last = {k: i for i, (k, _) in enumerate(records)}
    return [(i, k, v) for k, i in sorted(last.items(), key=lambda x: x[1]) for v in [records[i][1]] if v is not None]
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each record, check whether a later one has the same key. That costs O(n²) time and O(n) space."

"I can do better: Remember the last offset per key in one pass. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/log-compaction)

<a id="q19"></a>
### Q19. Why Queues Explode Near 100%

🟡 Interview standard · *Queueing theory*

For a single worker serving `mu` jobs/s (M/M/1 queue), return the utilisation % and average time in system (ms, rounded to 1) for each arrival rate — or 'unstable' when arrivals ≥ service rate.

**Examples**

- `mu = 100, lambdas = [50, 80, 90, 95, 99, 100, 120]` → `[(50.0, 20.0), (80.0, 50.0), (90.0, 100.0), (95.0, 200.0), (99.0, 1000.0), 'unstable', 'unstable']`

In [ ]:
def mm1(mu, lambdas):
    """
    For a single worker serving mu jobs/s (M/M/1 queue), return the utilisation
    % and average time in system (ms, rounded to 1) for each arrival rate — or
    'unstable' when arrivals ≥ service rate.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mm1, [
    ((100, [50, 80, 90, 95, 99, 100, 120]), [(50.0, 20.0), (80.0, 50.0), (90.0, 100.0), (95.0, 200.0), (99.0, 1000.0), 'unstable', 'unstable']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop with if/else.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop with if/else.

```python
def mm1(mu, lambdas):
    out = []
    for lam in lambdas:
        if lam >= mu:
            out.append("unstable")
        else:
            out.append((round(lam / mu * 100, 1), round(1000 / (mu - lam), 1)))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension.

```python
def mm1(mu, lambdas):
    return ["unstable" if l >= mu else (round(l / mu * 100, 1), round(1000 / (mu - l), 1)) for l in lambdas]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop with if/else. That costs O(n) time and O(n) space."

"I can do better: A comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/why-queues-explode-near-100)

<a id="q20"></a>
### Q20. Producer Batching (batch size and linger)

🟡 Interview standard · *Throughput vs latency*

A producer sends a batch when it reaches `max_batch` messages, or `linger` ms after the batch's first message — whichever comes first. Given message arrival times (ms, sorted), return `(send time, batch size)` for each batch.

**Examples**

- `times = [0, 1, 2, 3, 4, 20, 22, 50, 51, 52, 53, 54, 55], max_batch = 5, linger = 10` → `[(4, 5), (30, 2), (54, 5), (65, 1)]`

In [ ]:
def batching(times, max_batch, linger):
    """
    A producer sends a batch when it reaches max_batch messages, or linger ms
    after the batch's first message — whichever comes first. Given message
    arrival times (ms, sorted), return (send time, batch size) for each batch.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(batching, [
    (([0, 1, 2, 3, 4, 20, 22, 50, 51, 52, 53, 54, 55], 5, 10), [(4, 5), (30, 2), (54, 5), (65, 1)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate with a list for the open batch.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same with a start time and a counter.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Simulate with a list for the open batch.

```python
def batching(times, max_batch, linger):
    out, batch = [], []
    for t in times:
        if batch and t >= batch[0] + linger:
            out.append((batch[0] + linger, len(batch)))
            batch = []
        batch.append(t)
        if len(batch) == max_batch:
            out.append((t, len(batch)))
            batch = []
    if batch:
        out.append((batch[0] + linger, len(batch)))
    return out
```

**Cost:** O(n) time, O(b) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same with a start time and a counter.

```python
def batching(times, max_batch, linger):
    out, start, size = [], None, 0
    for t in times:
        if size and t >= start + linger:
            out.append((start + linger, size))
            size = 0
        if size == 0:
            start = t
        size += 1
        if size == max_batch:
            out.append((t, size))
            size = 0
    if size:
        out.append((start + linger, size))
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate with a list for the open batch. That costs O(n) time and O(b) space."

"I can do better: Same with a start time and a counter. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/system-design/#/q/queues/producer-batching-batch-size-and-linger)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. How do you guarantee ordering?</b></summary>

Only within a partition: key messages by the entity whose order matters (order id, user id). Global ordering needs a single partition, which limits throughput.

</details>

<details>
<summary><b>2. What happens if a consumer is too slow?</b></summary>

Lag grows. Add consumers (up to the partition count), add partitions, batch work, or make processing faster; alert on lag.

</details>

<details>
<summary><b>3. How do you handle a message that always fails?</b></summary>

Retry a limited number of times with backoff, then move it to a dead-letter queue, alert, and fix or replay it later — so it doesn't block the rest.

</details>

<details>
<summary><b>4. Why use the outbox pattern?</b></summary>

To change the database and publish an event atomically without distributed transactions: both go into one local transaction, and a relay publishes reliably afterwards.

</details>

<details>
<summary><b>5. Queue or direct API call?</b></summary>

Call directly when the caller needs the answer now. Use a queue when work can happen later, needs retries, or must absorb spikes.

</details>

<details>
<summary><b>6. Why does latency explode as utilisation approaches 100%?</b></summary>

Random arrivals cluster; with no spare capacity the queue can't catch up, so waits grow like 1 / (1 − utilisation). Keep busy resources well below 100%.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why put a queue in the middle?
- [ ] I can explain: Kafka in one lesson
- [ ] I can explain: Delivery guarantees and failure handling
- [ ] I can explain: The outbox pattern and stream processing

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Drop Late Events With a Watermark
- [ ] [Q17](#q17) A Delay Queue
- [ ] [Q18](#q18) Log Compaction
- [ ] [Q19](#q19) Why Queues Explode Near 100%
- [ ] [Q20](#q20) Producer Batching (batch size and linger)

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.